# 🤖 Aupa — Chatbot RAG de Recomendaciones Turísticas en Euskadi
### Bootcamp BBK The Bridge · Reto Inetum · Equipo 4
**Autores:** Naia, Andoni, Unai, Fátima  
**Fecha:** Junio 2026  
**Dataset:** `aupa_master_v7.csv` — Open Data Euskadi (~4.600 lugares)

---

## ¿Qué es este notebook?

Documenta la implementación completa del chatbot conversacional de **Aupa**, una app de recomendaciones turísticas auténticas para Euskadi.

El chatbot permite consultas en lenguaje natural como:
- *"Recomiéndame un restaurante de pintxos cerca del Guggenheim"*
- *"Playas tranquilas en Gipuzkoa"*
- *"Algo auténtico y local en Bilbao, sin turistas"*
- *"I want to visit a museum in San Sebastián"*

---

## Índice

1. [¿Qué es RAG y por qué lo usamos?](#1-rag)
2. [Arquitectura del sistema](#2-arquitectura)
3. [Instalación de dependencias](#3-dependencias)
4. [FASE 1 — Embeddings y construcción del índice FAISS](#4-faiss)
5. [FASE 2 — Backend FastAPI](#5-backend)
6. [FASE 3 — Mejoras de retrieval](#6-mejoras)
7. [El system prompt](#7-prompt)
8. [Elección y evolución del modelo de embeddings](#8-modelo)
9. [Problemas encontrados y soluciones](#9-bugs)
10. [Limitaciones conocidas y trabajo futuro](#10-limitaciones)


## 1. ¿Qué es RAG y por qué lo usamos? <a name="1-rag"></a>

### El problema

Queremos un chatbot que responda preguntas sobre lugares turísticos de Euskadi usando **nuestra propia base de datos**.

| Opción | ¿Por qué no? |
|--------|-------------|
| Entrenar modelo desde cero | Requiere millones de ejemplos, semanas de cómputo y cientos de miles de euros |
| Fine-tuning | Datos estáticos, se queda desactualizado, sigue siendo costoso |
| **RAG ✅** | Económico, actualizable, no requiere entrenamiento |

### ¿Qué significa RAG?

**RAG = Retrieval-Augmented Generation** (Generación Aumentada por Recuperación)

```
Usuario: "pintxos cerca del Guggenheim"
        ↓
[1] Nuestro código busca en el índice vectorial (BD) los lugares semánticamente más relevantes
        ↓
[2] Nuestro código construye un prompt con esos lugares
        ↓
[3] El LLM genera una respuesta en lenguaje natural
        ↓
"Te recomiendo el Bar Txiriboga, a 200m del Guggenheim..."
```

**Clave conceptual:** el LLM no consulta nuestra base de datos directamente. Es nuestro código el que orquesta todo. El LLM solo recibe texto y genera texto.

### ¿Qué es un embedding?

Un embedding es una representación numérica de un texto como un vector de números.

```
"restaurante de pintxos en Bilbao" → [0.23, -0.45, 0.12, 0.87, ...]  (384 números)
"bar de pintxos bilbaíno"          → [0.21, -0.43, 0.14, 0.85, ...]  (muy similar ✅)
"playa en Zarautz"                 → [0.67,  0.12, -0.34, 0.23, ...]  (muy diferente ✅)
```

Textos con significado similar tienen vectores similares. Esto permite búsqueda semántica: encontrar lugares relevantes aunque no compartan palabras exactas con la consulta.


## 2. Arquitectura del sistema <a name="2-arquitectura"></a>

### Fase offline (se ejecuta una vez al cambiar datos)

```
aupa_master_v7.csv  ──┐
                       ├──► build_faiss_index.py ──► faiss_all.index
txoko_reviews_raw.json ┘                          ├──► faiss_bizkaia.index
                                                  ├──► faiss_gipuzkoa.index
                                                  ├──► faiss_araba.index
                                                  └──► faiss_metadata_*.json
```

### Fase online (cada consulta del usuario)

```
Pregunta del usuario
        ↓
langdetect → detectar idioma
        ↓
Detectar territorio / municipio / landmark / categoría
        ↓
Seleccionar índice FAISS correcto (bizkaia / gipuzkoa / araba / all)
        ↓
encode_query() → embedding normalizado (384 dims)
        ↓
FAISS search k=50 candidatos
        ↓
Filtros post-FAISS: categoría → municipio → reordenar por proximidad
        ↓
build_prompt() → system prompt + lang_instruction + lugares + local_ratio
        ↓
Groq API (Llama 3.3 70B) → respuesta en lenguaje natural
        ↓
Usuario recibe respuesta
```

### Stack tecnológico

| Componente | Tecnología | Motivo |
|------------|-----------|--------|
| Embeddings | `sentence-transformers` | Open source, multilingüe, CPU |
| Índice vectorial | FAISS `IndexFlatIP` | Sin servidor, fichero en disco |
| LLM | Groq + Llama 3.3 70B | Tier gratuito generoso, sin restricciones geográficas |
| Backend | FastAPI | Ligero, fácil despliegue en Render |
| Detección idioma | `langdetect` | Multilingüe: es/eu/en |
| Despliegue | Render | Ya usado para BD y APIs del proyecto |


## 3. Instalación de dependencias <a name="3-dependencias"></a>

### ⚠️ Gestión segura de API keys

**Nunca escribas tu API key en el código ni en un chat.**

Crea un fichero `.env` en la carpeta del proyecto:
```
GROQ_API_KEY=tu_clave_aqui
```

Añade `.env` al `.gitignore`:
```bash
echo ".env" >> .gitignore
```


In [ ]:
# Instalar todas las dependencias
# Ejecutar una sola vez con el entorno virtual activo

# Crear entorno virtual (si no existe):
# python -m venv venv
# source venv/bin/activate        (Mac/Linux)
# venv\Scripts\activate          (Windows PowerShell)

!pip install sentence-transformers faiss-cpu pandas numpy
!pip install python-dotenv groq fastapi uvicorn langdetect
print("✓ Dependencias instaladas")


## 4. FASE 1 — Embeddings y construcción del índice FAISS <a name="4-faiss"></a>

### ¿Qué hace este paso?

1. Carga `aupa_master_v7.csv` y `txoko_reviews_raw.json`
2. Excluye registros marcados con `excluir_modelo = True` (fichas editoriales de municipios que generan ruido semántico)
3. Para cada lugar construye un texto combinando campos + reseñas
4. Genera un embedding por lugar (vector de 384 números -inicialmente-)
5. Construye 4 índices FAISS: uno global y uno por territorio
6. Guarda índices y metadatos en disco

### ¿Por qué índices separados por territorio?

Con un único índice global, una búsqueda de "pintxos en Bilbao" puede devolver resultados de Donostia o Vitoria porque FAISS solo entiende similitud semántica, no geografía. Con índices separados, cuando la query menciona Bizkaia (o un landmark de Bilbao), buscamos directamente en el índice de Bizkaia y es imposible que aparezcan resultados de otro territorio.

### ¿Por qué incorporamos reseñas al embedding?

La descripción oficial: *"Restaurante de cocina vasca tradicional"*  
Una reseña real: *"Las croquetas son increíbles, perfecto para una cena tranquila en familia"*

El segundo texto responde mejor a consultas como "cena tranquila" o "comida casera". Solo incorporamos reseñas con rating ≥ 3 para evitar que vocabulario negativo quede asociado al lugar.


In [ ]:
import json, os, time
import faiss
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer

# ─── Configuración ────────────────────────────────────────────────────────────
CSV_PATH     = "aupa_master_v7.csv"
REVIEWS_PATH = "txoko_reviews_raw.json"
OUTPUT_DIR   = "./rag_assets"

# Modelo de embeddings — ver sección 8 para justificación de la elección
MODEL_NAME   = "paraphrase-multilingual-mpnet-base-v2"
MAX_REVIEWS  = 5

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"✓ Configuración lista | Modelo: {MODEL_NAME}")


In [ ]:
# ─── Funciones de construcción de texto ──────────────────────────────────────

def load_reviews(path):
    with open(path, encoding="utf-8") as f:
        raw = json.load(f)
    reviews = {}
    for pid, data in raw.items():
        if isinstance(data, dict) and "reviews" in data:
            reviews[pid] = data["reviews"]
        elif isinstance(data, list):
            reviews[pid] = data
    print(f"  Reseñas cargadas: {len(reviews):,} lugares")
    return reviews

def build_place_text(row, reviews_by_id):
    """
    Construye el texto que se convertirá en embedding para un lugar.
    Combina campos estructurados del CSV con reseñas reales de Google.
    """
    parts = [
        f"Nombre: {row.get('nombre', '')}",
        f"Tipo: {row.get('subcategoria', '')} | {row.get('categoria', '')}",
        f"Ubicación: {row.get('municipio', '')}, {row.get('territorio', '')}",
    ]
    desc = str(row.get("descripcion", "")).strip()
    if desc and desc != "nan":
        parts.append(f"Descripción: {desc}")

    place_id = str(row.get("id", ""))
    if place_id in reviews_by_id:
        texts = [
            r["text"].strip()
            for r in reviews_by_id[place_id][:MAX_REVIEWS]
            if r.get("text", "").strip() and r.get("rating", 0) >= 3
        ]
        if texts:
            parts.append("Reseñas: " + " | ".join(texts))
    return "\n".join(parts)

def build_metadata(row):
    def safe(v):
        return None if pd.isna(v) else v
    return {
        "id":                 str(row.get("id", "")),
        "nombre":             str(row.get("nombre", "")),
        "categoria":          str(row.get("categoria", "")),
        "subcategoria":       str(row.get("subcategoria", "")),
        "municipio":          str(row.get("municipio", "")),
        "territorio":         str(row.get("territorio", "")),
        "lat":                safe(row.get("lat")),
        "lon":                safe(row.get("lon")),
        "descripcion":        str(row.get("descripcion", "")),
        "google_rating":      safe(row.get("google_rating")),
        "google_num_reviews": safe(row.get("google_num_reviews")),
        "local_ratio":        safe(row.get("local_ratio")),
        "web":                str(row.get("web", "")),
        "ficha_turismo":      str(row.get("ficha_turismo", "")),
    }

print("✓ Funciones definidas")


In [ ]:
# ─── 1. Cargar y filtrar datos ───────────────────────────────────────────────

print("[1/5] Cargando datos...")
df = pd.read_csv(CSV_PATH)
print(f"  CSV cargado: {len(df):,} registros")

# Excluir fichas editoriales de municipios (campo excluir_modelo)
# Estas entradas describen pueblos enteros con texto muy genérico
# que confunde al modelo semántico y aparece en búsquedas de cualquier tipo
antes = len(df)
df = df[df["excluir_modelo"] != True]
print(f"  Excluidos por excluir_modelo: {antes - len(df)} registros")

df = df[df["nombre"].notna() & df["nombre"].str.strip().ne("")]
print(f"  Registros para indexar: {len(df):,}")

reviews_by_id = load_reviews(REVIEWS_PATH)


In [ ]:
# ─── 2-3. Construir textos y generar embeddings ──────────────────────────────

print("\n[2/5] Construyendo textos...")
texts    = [build_place_text(row, reviews_by_id) for _, row in df.iterrows()]
metadata = [build_metadata(row) for _, row in df.iterrows()]

n_con_reviews = sum(1 for m in metadata if m["id"] in reviews_by_id)
print(f"  Con reseñas incorporadas: {n_con_reviews:,} / {len(texts):,}")
print(f"\n  Ejemplo texto primer lugar:")
print("  " + "\n  ".join(texts[0].split("\n")))

print(f"\n[3/5] Cargando modelo '{MODEL_NAME}'...")
model = SentenceTransformer(MODEL_NAME)
print(f"  Dimensión de embeddings: {model.get_sentence_embedding_dimension()}")

print(f"\n  Generando embeddings para {len(texts):,} textos (puede tardar varios minutos en CPU)...")
t0 = time.time()
embeddings = model.encode(
    texts, batch_size=64, show_progress_bar=True, normalize_embeddings=True
)
print(f"  Completado en {time.time()-t0:.1f}s | Shape: {embeddings.shape}")


In [ ]:
# ─── 4-5. Construir y guardar índices FAISS ──────────────────────────────────

def build_and_save_index(emb, meta, name, output_dir):
    """Construye un índice FAISS y guarda índice + metadatos en disco."""
    dim   = emb.shape[1]
    index = faiss.IndexFlatIP(dim)
    index.add(emb.astype(np.float32))
    
    idx_path  = os.path.join(output_dir, f"faiss_{name}.index")
    meta_path = os.path.join(output_dir, f"faiss_metadata_{name}.json")
    
    faiss.write_index(index, idx_path)
    with open(meta_path, "w", encoding="utf-8") as f:
        json.dump(meta, f, ensure_ascii=False, indent=2)
    
    print(f"  ✓ faiss_{name}.index — {index.ntotal:,} vectores ({os.path.getsize(idx_path)//1024} KB)")
    return index

print("[4/5] Construyendo índices FAISS...")

# Índice global
build_and_save_index(embeddings, metadata, "all", OUTPUT_DIR)

# Índices por territorio
TERRITORIOS = ["BIZKAIA", "GIPUZKOA", "ARABA"]
for territorio in TERRITORIOS:
    mask   = df["territorio"].str.upper() == territorio
    df_t   = df[mask]
    emb_t  = embeddings[mask.values]
    meta_t = [m for m, keep in zip(metadata, mask) if keep]
    if len(df_t) > 0:
        build_and_save_index(emb_t, meta_t, territorio.lower(), OUTPUT_DIR)

print(f"\n[5/5] ✓ Índices generados en {OUTPUT_DIR}/")
print("  Siguiente paso: arrancar el backend FastAPI")


In [ ]:
# ─── Verificación rápida del índice ──────────────────────────────────────────

index_all = faiss.read_index(os.path.join(OUTPUT_DIR, "faiss_all.index"))
with open(os.path.join(OUTPUT_DIR, "faiss_metadata_all.json"), encoding="utf-8") as f:
    metadata_all = json.load(f)

def buscar(pregunta, k=5):
    print(f"\nPregunta: '{pregunta}'")
    print("-" * 55)
    emb = model.encode([pregunta], normalize_embeddings=True)
    scores, indices = index_all.search(emb.astype(np.float32), k)
    for score, idx in zip(scores[0], indices[0]):
        lugar = metadata_all[idx]
        print(f"  [{score:.3f}] {lugar['nombre']} — {lugar['municipio']} ({lugar['categoria']})")

# Pruebas de validación
buscar("restaurante pintxos Bilbao")
buscar("playa tranquila Gipuzkoa")
buscar("museo arte contemporáneo")
buscar("alojamiento rural campo")


## 5. FASE 2 — Backend FastAPI <a name="5-backend"></a>

El backend es el servidor que orquesta el pipeline completo en tiempo real. Está estructurado en módulos para que cada parte sea fácil de entender y modificar.

### Estructura de ficheros

```
backend/
├── main.py              ← servidor FastAPI, endpoint /chat
├── requirements.txt
├── .env                 ← API keys (nunca al repositorio)
└── rag/
    ├── config.py        ← configuración centralizada
    ├── index_loader.py  ← carga de índices FAISS al arrancar
    ├── retrieval.py     ← pipeline de búsqueda con filtros
    ├── prompt.py        ← construcción del prompt
    ├── llm.py           ← llamada a Groq API
    └── landmarks.py     ← coordenadas de ~60 puntos de referencia
```

### Elección del LLM: de Gemini a Groq

Durante el desarrollo encontramos problemas con Gemini en el tier gratuito:

| LLM probado | Problema |
|-------------|----------|
| `gemini-2.5-flash` | Cuota diaria muy baja, se agota en pocas pruebas |
| `gemini-2.0-flash` | `limit: 0` desde España — restricción geográfica no documentada |
| `gemini-1.5-flash` | Modelo no disponible en API v1beta |
| **Groq + Llama 3.3 70B ✅** | Tier gratuito generoso, sin restricciones geográficas |

**Ventaja adicional de Groq:** el system prompt va en un mensaje con `role: system` separado del contenido del usuario, lo que garantiza que el LLM lo trata con prioridad real.


In [ ]:
# ── config.py ─────────────────────────────────────────────────────────────────
# Muestra del contenido — no ejecutar directamente

CONFIG = '''
import os
from pathlib import Path

BASE_DIR = Path(__file__).resolve().parent.parent.parent  # raíz del proyecto
RAG_DIR  = BASE_DIR / "rag_assets"

# Rutas de índices
FAISS_INDEXES = {
    "all":      RAG_DIR / "faiss_all.index",
    "bizkaia":  RAG_DIR / "faiss_bizkaia.index",
    "gipuzkoa": RAG_DIR / "faiss_gipuzkoa.index",
    "araba":    RAG_DIR / "faiss_araba.index",
}
FAISS_METADATA = {
    "all":      RAG_DIR / "faiss_metadata_all.json",
    "bizkaia":  RAG_DIR / "faiss_metadata_bizkaia.json",
    "gipuzkoa": RAG_DIR / "faiss_metadata_gipuzkoa.json",
    "araba":    RAG_DIR / "faiss_metadata_araba.json",
}

# Modelo de embeddings — debe coincidir con el usado para generar el índice
EMBEDDING_MODEL = "paraphrase-multilingual-mpnet-base-v2"

# Retrieval
FAISS_TOP_K = 50   # candidatos recuperados de FAISS
FINAL_TOP_K = 5    # lugares enviados al LLM

# LLM
GROQ_API_KEY = os.getenv("GROQ_API_KEY", "")
GROQ_MODEL   = "llama-3.3-70b-versatile"
'''
print(CONFIG)


In [ ]:
# ── index_loader.py ───────────────────────────────────────────────────────────

INDEX_LOADER = '''
import json, faiss
from pathlib import Path
from .config import FAISS_INDEXES, FAISS_METADATA

def load_indexes():
    """
    Carga todos los índices FAISS al arrancar el servidor.
    Se ejecuta UNA SOLA VEZ en el lifespan de FastAPI.
    
    IMPORTANTE: --reload de uvicorn recarga el código Python pero NO
    los ficheros .index. Tras regenerar el índice hay que reiniciar
    uvicorn manualmente (Ctrl+C y volver a arrancar).
    """
    indexes = {}
    for name, path in FAISS_INDEXES.items():
        if path.exists():
            indexes[name] = {
                "index":    faiss.read_index(str(path)),
                "metadata": json.load(open(FAISS_METADATA[name], encoding="utf-8"))
            }
            print(f"  ✓ Índice {name}: {indexes[name]['index'].ntotal:,} vectores")
        else:
            print(f"  ⚠ Índice {name} no encontrado: {path}")
    return indexes
'''
print(INDEX_LOADER)


In [ ]:
# ── llm.py ────────────────────────────────────────────────────────────────────

LLM_CODE = '''
from groq import Groq
from .config import GROQ_API_KEY, GROQ_MODEL

_client = None

def init_llm():
    global _client
    if not GROQ_API_KEY:
        raise EnvironmentError("GROQ_API_KEY no encontrada en .env")
    _client = Groq(api_key=GROQ_API_KEY)

def call_llm(prompt: str, system_prompt: str) -> str:
    """
    Llama a Groq con el prompt del usuario y el system prompt separados.
    
    La separación de roles es clave: con role="system" el LLM trata
    esas instrucciones con prioridad, lo que mejora el seguimiento
    de instrucciones como el idioma de respuesta.
    """
    response = _client.chat.completions.create(
        model=GROQ_MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": prompt},
        ],
        temperature=0.7,
        max_tokens=1024,
    )
    return response.choices[0].message.content
'''
print(LLM_CODE)


In [ ]:
# ── main.py ───────────────────────────────────────────────────────────────────

MAIN_CODE = '''
from contextlib import asynccontextmanager
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from sentence_transformers import SentenceTransformer

from rag.config import EMBEDDING_MODEL
from rag.index_loader import load_indexes
from rag.llm import call_llm, init_llm
from rag.prompt import build_prompt
from rag.retrieval import retrieve

class AppState:
    indexes:  dict
    model:    SentenceTransformer

state = AppState()

@asynccontextmanager
async def lifespan(app: FastAPI):
    # Carga al arrancar — solo una vez
    print("⏳ Cargando modelo de embeddings...")
    state.model = SentenceTransformer(EMBEDDING_MODEL)
    print("⏳ Cargando índices FAISS...")
    state.indexes = load_indexes()
    init_llm()
    print("✅ Aupa RAG listo")
    yield

app = FastAPI(title="Aupa RAG API", lifespan=lifespan)

class ChatRequest(BaseModel):
    query: str

class ChatResponse(BaseModel):
    answer: str

@app.get("/health")
def health():
    return {"status": "ok", "indexes": list(state.indexes.keys())}

@app.post("/chat")
def chat(request: ChatRequest) -> ChatResponse:
    query = request.query.strip()
    if not query:
        raise HTTPException(status_code=422, detail="La query no puede estar vacía.")
    
    places = retrieve(query, state.model, state.indexes)
    if not places:
        return ChatResponse(answer="No he encontrado lugares relevantes para tu consulta.")
    
    prompt, system = build_prompt(query, places)
    try:
        answer = call_llm(prompt, system)
    except Exception as e:
        raise HTTPException(status_code=502, detail=f"Error LLM: {e}")
    
    return ChatResponse(answer=answer)
'''
print(MAIN_CODE)


## 6. FASE 3 — Mejoras de retrieval <a name="6-mejoras"></a>

FAISS devuelve los k lugares más similares semánticamente, pero eso no siempre es suficiente. La búsqueda semántica no entiende geografía ni categorías. Añadimos una capa de filtros post-FAISS para corregirlo.

### El problema sin filtros

- "pintxos en Bilbao" → puede devolver restaurantes de Donostia
- "ruta en Gipuzkoa" → puede devolver rutas de Bizkaia  
- "hotel cerca del Guggenheim" → puede devolver un hotel en Santurtzi a 8km

### La solución: índices territoriales + filtros en cadena

```
FAISS devuelve k=50 candidatos del índice territorial correcto
        ↓
Filtro 1: categoría (si se puede inferir de la query)
        ↓
Filtro 2a: landmark → filtrar por municipio + reordenar por distancia haversine
Filtro 2b: municipio explícito → filtrar
        ↓
Devolver FINAL_TOP_K mejores
```

### Landmarks: puntos de referencia turísticos

Diccionario con ~60 puntos de Bilbao, Donostia y Vitoria-Gasteiz con sus coordenadas GPS. Cuando la query menciona un landmark, los candidatos se reordenan por distancia real (haversine) y el LLM puede decir "a 350m del Guggenheim" con datos reales (```landmarks.py```)

```python
LANDMARKS = {
    "guggenheim":     (43.2686, -2.9340),   # Bilbao → BIZKAIA
    "casco viejo":    (43.2587, -2.9236),
    "san mamés":      (43.2643, -2.9494),
    "parte vieja":    (43.3224, -1.9847),   # Donostia → GIPUZKOA
    "la concha":      (43.3182, -1.9860),
    "kursaal":        (43.3233, -1.9765),
    "artium":         (42.8475, -2.6676),   # Vitoria → ARABA
    "virgen blanca":  (42.8467, -2.6725),
    # ... ~60 landmarks en total
}
```

### Detección de categoría

Cuando la query menciona un tipo de lugar, se filtra por categoría antes de pasar los candidatos al LLM. Esto evita que "restaurante de pintxos" devuelva hoteles.

```python
CATEGORIA_KEYWORDS = {
    ("Culinario",):            ["comer", "cenar", "restaurante", "pintxos", "sidrería", "bar"],
    ("Alojamiento",):          ["dormir", "hotel", "alojamiento", "hostal", "camping"],
    ("Cultural",):             ["museo", "arte", "cultura", "patrimonio"],
    ("Naturaleza", "Servicios"): ["ruta", "rutas", "senderismo", "hiking", "paseo"],
    ("Naturaleza",):           ["playa", "beach", "parque natural"],
    ("Ocio",):                 ["actividad", "deporte", "surf", "kayak"],
}
```

Nota: "rutas" busca en **Naturaleza Y Servicios** porque Open Data Euskadi categoriza muchas rutas como "Servicios" en origen.


In [ ]:
# ── retrieval.py — función retrieve() completa ────────────────────────────────

RETRIEVAL_CODE = '''
from langdetect import detect, LangDetectException
from .landmarks import find_landmark, reorder_by_proximity, LANDMARK_A_MUNICIPIO
import re, numpy as np

TERRITORIOS = {
    "gipuzkoa": "GIPUZKOA", "guipúzcoa": "GIPUZKOA", "guipuzcoa": "GIPUZKOA",
    "bizkaia":  "BIZKAIA",  "vizcaya":   "BIZKAIA",
    "araba":    "ARABA",    "álava":     "ARABA",    "alava": "ARABA",
}

MUNICIPIOS = [
    "bilbao", "donostia", "san sebastián", "san sebastian", "vitoria", "gasteiz",
    "barakaldo", "getxo", "irun", "eibar", "zarautz", "hondarribia", "bermeo",
    "gernika", "lekeitio", "durango", "arrasate", "ondarroa", "zarautz",
]

CATEGORIA_KEYWORDS = {
    ("Culinario",):              ["comer", "cenar", "restaurante", "pintxos",
                                  "sidrería", "bar", "eat", "food", "dinner"],
    ("Alojamiento",):            ["dormir", "hotel", "alojamiento", "hostal",
                                  "sleep", "stay", "camping"],
    ("Cultural",):               ["museo", "arte", "cultura", "museum", "art"],
    ("Naturaleza", "Servicios"): ["ruta", "rutas", "senderismo", "hiking",
                                  "hike", "paseo"],
    ("Naturaleza",):             ["playa", "beach", "parque natural"],
    ("Ocio",):                   ["actividad", "deporte", "surf", "kayak"],
}

MUNICIPIO_A_TERRITORIO = {
    "bilbao": "BIZKAIA", "barakaldo": "BIZKAIA", "getxo": "BIZKAIA",
    "bermeo": "BIZKAIA", "gernika": "BIZKAIA",   "durango": "BIZKAIA",
    "donostia": "GIPUZKOA", "san sebastián": "GIPUZKOA", "irun": "GIPUZKOA",
    "zarautz": "GIPUZKOA",  "hondarribia": "GIPUZKOA",
    "vitoria": "ARABA",    "gasteiz": "ARABA",
}

def encode_query(query, model):
    vector = model.encode([query], convert_to_numpy=True, normalize_embeddings=True)
    return vector.astype("float32")

def _extract_territorio(query):
    q = query.lower()
    for term, terr in TERRITORIOS.items():
        if re.search(rf"\\b{re.escape(term)}\\b", q):
            return terr
    return None

def _extract_municipio(query):
    q = query.lower()
    for m in MUNICIPIOS:
        if re.search(rf"\\b{re.escape(m)}\\b", q):
            return m
    return None

def _extract_categoria(query):
    q = query.lower()
    for cats, keywords in CATEGORIA_KEYWORDS.items():
        if any(k in q for k in keywords):
            return list(cats)
    return None

def retrieve(query, model, indexes, k=50):
    vector     = encode_query(query, model)
    territorio = _extract_territorio(query)
    municipio  = _extract_municipio(query)
    landmark   = find_landmark(query)
    categoria  = _extract_categoria(query)

    # Inferir municipio desde landmark
    if landmark and not municipio:
        municipio = LANDMARK_A_MUNICIPIO.get(landmark[0])

    # Inferir territorio desde municipio
    if municipio and not territorio:
        territorio = MUNICIPIO_A_TERRITORIO.get(municipio)

    # Seleccionar índice territorial si está disponible
    idx_name = territorio.lower() if territorio and territorio.lower() in indexes else "all"
    idx      = indexes[idx_name]["index"]
    metadata = indexes[idx_name]["metadata"]

    distances, indices = idx.search(vector, k)
    candidates = []
    for dist, i in zip(distances[0], indices[0]):
        if i == -1: continue
        entry = metadata[i].copy()
        entry["_score"] = float(dist)
        candidates.append(entry)

    # Filtro 1: categoría obligatorio si se infiere
    if categoria:
        filtered = [c for c in candidates if c.get("categoria") in categoria]
        if len(filtered) >= 1:
            candidates = filtered

    # Filtro 2a: landmark → municipio + proximidad
    if landmark:
        _, coords = landmark[1], landmark[1]
        if municipio:
            in_mun = [c for c in candidates
                      if municipio in (c.get("municipio") or "").lower()]
            if in_mun:
                candidates = in_mun
        candidates = reorder_by_proximity(candidates, landmark[1])

    # Filtro 2b: municipio sin landmark
    elif municipio:
        filtered = [c for c in candidates
                    if municipio in (c.get("municipio") or "").lower()]
        if filtered:
            candidates = filtered

    return candidates[:FINAL_TOP_K]
'''
print(RETRIEVAL_CODE)


## 7. El system prompt <a name="7-prompt"></a>

El system prompt define el comportamiento permanente del chatbot. Es la pieza donde más impacto tiene el diseño sobre la calidad de las respuestas.

### Decisiones de diseño

**¿Por qué la instrucción de idioma en inglés al principio?**  
Los LLMs dan más peso a las instrucciones al inicio del prompt. Escribirla en inglés ayuda porque los modelos tienen más datos de entrenamiento en inglés. Aun así, con modelos pequeños esto no es suficiente — necesitamos reforzarlo dinámicamente en cada consulta con `langdetect`.

**Detección de idioma con langdetect:**
```python
from langdetect import detect

lang = detect(query)  # "es", "en", "eu"...
if lang == "en":
    lang_instruction = "IMPORTANT: The user is writing in English. You MUST respond in English."
elif lang == "eu":
    lang_instruction = "GARRANTZITSUA: Erabiltzaileak euskaraz idazten du. Erantzun euskaraz."
else:
    lang_instruction = "IMPORTANTE: El usuario escribe en español. Responde en español."
```

**¿Por qué "entre 1 y 3 recomendaciones"?**  
"Máximo 3" hacía que el LLM forzara una tercera recomendación aunque no encajara. "Entre 1 y 3 según información disponible" produce respuestas más honestas.

**La instrucción de autenticidad local:**  
El campo `local_ratio` del CSV (0-1, escalado a /100) mide qué tan frecuentado es un lugar por locales vs turistas. Se incluye en el contexto de cada lugar y el LLM lo prioriza cuando el usuario pide algo "auténtico" o "sin turistas". Los valores máximos en el dataset son ~70/100, que ya indica un lugar genuinamente local.


In [ ]:
# ── System prompt completo ────────────────────────────────────────────────────

SYSTEM_PROMPT = (
    "CRITICAL INSTRUCTION: Always respond in the same language the user writes in. "
    "If the user writes in English, respond in English. "
    "If the user writes in Spanish, respond in Spanish. "
    "If the user writes in Basque (Euskera), respond in Basque. "
    "Never respond in a different language than the one the user used.\n\n"
    "Eres un asistente de recomendaciones turísticas de Euskadi llamado Aupa.\n"
    "Usa únicamente la información proporcionada en el contexto.\n"
    "No inventes datos como horarios, precios ni distancias.\n"
    "Si no tienes suficiente información, dilo claramente.\n"
    "Si los lugares no encajan bien con la pregunta, dilo explícitamente.\n"
    "Si la pregunta es geográfica como 'cerca de X', advierte que no puedes "
    "garantizar proximidad exacta y sugiere verificar en el mapa.\n"
    "Presenta entre 1 y 3 recomendaciones según la información disponible. "
    "Si solo tienes 1 o 2 opciones relevantes, preséntelas sin forzar más. "
    "No recomiendes un lugar si no encaja claramente con la consulta.\n"
    "Si el usuario pide algo auténtico, local o alejado del turismo masivo, "
    "prioriza los lugares con mayor puntuación de autenticidad local. "
    "Una puntuación de 60/100 o más indica un lugar genuinamente local.\n"
    "Sé claro, útil y ligeramente cercano en tono.\n"
)

print("System prompt definido:")
print(SYSTEM_PROMPT)


## 8. Elección y evolución del modelo de embeddings <a name="8-modelo"></a>

### Comparativa de modelos evaluados

| Modelo | Dims | Tamaño | CPU (~4.600 docs) | Requiere prefijos |
|--------|------|--------|-------------------|-------------------|
| `paraphrase-multilingual-MiniLM-L12-v2` | 384 | 118MB | ~2.5 min | No |
| `paraphrase-multilingual-mpnet-base-v2` ✅ | 768 | 278MB | ~8 min | No |
| `intfloat/multilingual-e5-large` | 1024 | 560MB | ~25 min | Sí (`query:` / `passage:`) |

### Por qué empezamos con MiniLM

Punto de partida natural: multilingüe, ligero, sin GPU, bien documentado. Scores iniciales razonables (0.70-0.80 para consultas claras).

### Por qué migramos a mpnet

Al comparar los scores entre modelos para las mismas consultas, mpnet mostraba mejoras consistentes:

| Consulta | MiniLM | mpnet |
|----------|--------|-------|
| "restaurante Bilbao" | ~0.75 | 0.821 |
| "cerca del Guggenheim" | ~0.62 | 0.691 |
| "hotel rural Bizkaia" | ~0.72 | 0.782 |

La mejora es especialmente notable en consultas semánticas complejas. El coste: el índice pasa de ~7MB a ~14MB y la generación tarda ~8 min en CPU. Ambos son aceptables para nuestro caso de uso.

### Por qué descartamos multilingual-e5-large

`e5-large` con 1024 dimensiones y ~560MB presenta dos problemas para nuestro despliegue:

1. **Memoria en Render:** el tier gratuito tiene 512MB de RAM. Cargar el modelo + el índice supera ese límite.
2. **Tiempo de generación:** ~25 minutos en CPU para regenerar los 5 índices hace inviable iterar durante el desarrollo.

Queda como **mejora futura** si se dispone de un tier de Render con más memoria o de una GPU para la generación offline.

### Justificación de mpnet

**Multilingüe:** un único espacio vectorial para español, euskera e inglés. "pintxos bar in Bilbao" y "bar de pintxos en Bilbao" producen embeddings cercanos.

**Arquitectura mpnet vs MiniLM:** mpnet usa una arquitectura de transformers más profunda (base vs mini) que captura mejor las relaciones semánticas complejas. Especialmente relevante para consultas con múltiples conceptos como "alojamiento rural tranquilo con vistas al mar en Gipuzkoa".

**Sin dependencias externas en indexación:** el modelo corre en local, el índice es reproducible sin coste y no hay dependencia de APIs de pago en la fase de indexación. Contrasta con alternativas como OpenAI `text-embedding-ada-002` que requiere API de pago por cada regeneración del índice.


## 9. Problemas encontrados y soluciones <a name="9-bugs"></a>

Esta sección documenta los bugs reales que encontramos durante el desarrollo. Es parte honesta e importante del proceso.

### 9.1 Scores anómalos en FAISS (> 1.0)

**Síntoma:** scores de 2.67, 2.75... cuando deberían estar entre 0 y 1.

**Causa:** en `encode_query()` faltaba `normalize_embeddings=True`. Los vectores de las queries no estaban normalizados, mientras que los del índice sí. Eso rompía la similitud coseno.

```python
# MAL
vector = model.encode([query], convert_to_numpy=True)

# BIEN — normalize_embeddings debe ser True en indexación Y en consulta
vector = model.encode([query], convert_to_numpy=True, normalize_embeddings=True)
```

**Lección:** el modelo debe usarse exactamente igual en indexación y en consulta. Cualquier diferencia rompe la comparabilidad.

---

### 9.2 El backend cargaba el índice antiguo

**Síntoma:** después de regenerar el índice los scores no mejoraban.

**Causa:** `--reload` de uvicorn recarga ficheros `.py` pero no ficheros `.index`.

**Solución:** parar y arrancar uvicorn manualmente tras regenerar el índice.

---

### 9.3 El LLM respondía siempre en castellano

**Causa raíz:** dos problemas encadenados:
1. Con Gemini: el `system_instruction` se mezclaba con el contenido como texto plano
2. Con Groq/Llama: modelos pequeños ignoran instrucciones de idioma en el system prompt cuando este está en castellano

**Solución:** detectar idioma con `langdetect` e inyectarlo en el mensaje del usuario:
```python
lang = detect(query)
lang_instruction = "IMPORTANT: Respond in English." if lang == "en" else "IMPORTANTE: Responde en español."
user_section = f"{lang_instruction}\n\n{contenido}"
```

---

### 9.4 Rutas categorizadas como "Servicios"

**Síntoma:** búsquedas de rutas devolvían resultados irrelevantes porque el filtro de categoría buscaba `Naturaleza` pero Open Data Euskadi categoriza muchas rutas como `Servicios`.

**Solución:** el filtro de categoría para "ruta/senderismo" busca en ambas:
```python
("Naturaleza", "Servicios"): ["ruta", "rutas", "senderismo", "hiking"]
```

---

### 9.5 Fichas editoriales de municipios como ruido semántico

**Síntoma:** búsquedas de cualquier tipo devolvían entradas de pueblos (Berriatua, Otxandio...) que no son lugares turísticos concretos sino descripciones genéricas de municipios.

**Causa:** Open Data Euskadi incluye fichas editoriales de municipios con texto muy largo y genérico que menciona todo tipo de recursos. El embedding de esos textos es "similar a todo".

**Solución:** el CSV tiene un campo `excluir_modelo = True` que marca estas fichas. Se filtran antes de generar el índice:
```python
df = df[df["excluir_modelo"] != True]
```


## 10. Limitaciones conocidas y trabajo futuro <a name="10-limitaciones"></a>

### Limitaciones actuales

| Limitación | Causa | Solución futura |
|------------|-------|----------------|
| Sin horarios ni precios | Open Data Euskadi no los publica | Integrar Google Places Details API |
| Consultas muy vagas dan resultados dispares | FAISS necesita contexto semántico rico | Mejorar descripciones en CSV |
| "Actividades para niños" funciona mal | Categoría no etiquetada en datos | Etiquetado manual o taxonomía propia |
| Nombres propios poco conocidos | No están en textos embeddados | Búsqueda híbrida BM25 + embeddings |
| Eventos en tiempo real | No están en Open Data Euskadi | API agenda Bilbao / Eventbrite |
| e5-large descartado por memoria | Render tier gratuito 512MB RAM | Tier de pago o GPU para indexación |

### Qué funciona bien

- Consultas con categoría + ubicación: "restaurante en Bilbao", "museo en Vitoria" ✅
- Consultas con landmark: "cerca del Guggenheim", "junto a la Parte Vieja" ✅
- Consultas de autenticidad: "algo local sin turistas" ✅
- Multilingüe: español, inglés, euskera ✅
- Distancias reales a puntos de referencia ✅
- Respeta categorías: "pintxos" no devuelve hoteles ✅

### Mejoras técnicas pendientes

**Búsqueda híbrida (BM25 + embeddings):** combinar búsqueda por texto exacto con semántica. Mejoraría resultados para nombres propios específicos.

**Re-ranking con cross-encoder:** usar un modelo de re-ranking para reordenar candidatos con más precisión antes de pasarlos al LLM.

**Modelo multilingual-e5-large:** con infraestructura adecuada (>512MB RAM o GPU), este modelo de 1024 dimensiones podría mejorar los scores en consultas complejas respecto a mpnet.

**Evaluación sistemática:** construir un conjunto de 50-100 preguntas con respuesta esperada y medir la calidad automáticamente.

---

## Pipeline completo — resumen visual

```
[OFFLINE — ejecutar al cambiar datos]
aupa_master_v7.csv + txoko_reviews_raw.json
    → filtrar excluir_modelo=True
    → build_place_text() (campos + reseñas ≥3★)
    → paraphrase-multilingual-mpnet-base-v2
    → normalize_embeddings=True
    → IndexFlatIP × 4 (all, bizkaia, gipuzkoa, araba)
    → faiss_*.index + faiss_metadata_*.json

[ONLINE — cada consulta]
"pintxos cerca del Guggenheim"
    → langdetect → "es"
    → find_landmark("guggenheim") → BIZKAIA → faiss_bizkaia.index
    → encode_query() normalize=True → vector 768 dims
    → FAISS search k=50
    → _extract_categoria() → ["Culinario"] → filtrar
    → reorder_by_proximity(43.2686, -2.9340)
    → build_prompt(): lang_instruction + lugares + local_ratio + distancias
    → Groq Llama 3.3 70B (system + user separados)
    → "Te recomiendo Porrue, a 312m del Guggenheim (local_ratio: 62/100)..."
```
